In [29]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import (
    accuracy_score, 
    recall_score, 
    f1_score, 
    confusion_matrix, 
    roc_auc_score,
    roc_curve,
    classification_report
)
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline
import warnings
warnings.filterwarnings('ignore')

In [2]:
df = pd.read_csv("../Data/preprocessed.csv")
df.head()

,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,OnlineSecurity,OnlineBackup,DeviceProtection,...,PaperlessBilling,MonthlyCharges,TotalCharges,Churn,Average,InternetService_Fiber optic,InternetService_No,PaymentMethod_Credit card (automatic),PaymentMethod_Electronic check,PaymentMethod_Mailed check
0,0,0,1,0,1,0,0,0,1,0,...,1,29.85,29.85,0,29.850000,0,0,0,1,0
1,1,0,0,0,34,1,0,1,0,1,...,0,56.95,1889.50,0,55.573529,0,0,0,0,1
2,1,0,0,0,2,1,0,1,1,0,...,1,53.85,108.15,1,54.075000,0,0,0,0,1
3,1,0,0,0,45,0,0,1,0,1,...,0,42.30,1840.75,0,40.905556,0,0,0,0,0
4,0,0,0,0,2,1,0,0,0,0,...,1,70.70,151.65,1,75.825000,1,0,0,1,0


In [3]:
x = df.drop(['Churn'], axis=1)
y = df[['Churn']]                                     

In [4]:
xtrain, xtest, ytrain, ytest = train_test_split(
    x, y, 
    random_state=42,
    test_size=0.2,                                                                         
    stratify=y
)

In [5]:
xtrain_no_scale = xtrain.copy()
ytrain_no_scale = ytrain.copy()

In [6]:
smote = SMOTE(random_state=42)
xtrain_no_scale_res, ytrain_no_scale_res = smote.fit_resample(xtrain_no_scale, ytrain_no_scale)

# Applying without Scalling

In [7]:
algorithms = {
    "LogisticRegression": LogisticRegression(),
    "KNeighborsClassifier": KNeighborsClassifier(),
    "RandomForestClassifier": RandomForestClassifier(),
    "XGBClassifier": XGBClassifier(),
    "GaussianNB": GaussianNB()
}

In [ ]:
result_list = []

for model_name, model in algorithms.items():
    model.fit(xtrain_no_scale_res, ytrain_no_scale_res)
    pred = model.predict(xtest)

    result_list.append({
        'Model': model_name,
        'Train Score': model.score(xtrain_no_scale_res, ytrain_no_scale_res),
        'Test Score': model.score(xtest, ytest),
        'Accuracy Score': accuracy_score(ytest, pred),
        'Recall Score': recall_score(ytest, pred),
        'F1 Score': f1_score(ytest, pred),
        'Roc Auc Score': roc_auc_score(ytest, pred),
        'Confussion Matrix': confusion_matrix(ytest, pred)

    })

    results = pd.DataFrame(result_list)

In [23]:
results.sort_values(by='Roc Auc Score', ascending=False)

,Model,Train Score,Test Score,Accuracy Score,Recall Score,F1 Score,Roc Auc Score,Confussion Matrix
0,LogisticRegression,0.819855,0.751244,0.751244,0.689840,0.595843,0.731657,"[[799, 234], [116, 258]]"
2,RandomForestClassifier,0.999153,0.778252,0.778252,0.628342,0.601023,0.730434,"[[860, 173], [139, 235]]"
4,GaussianNB,0.805690,0.749112,0.749112,0.681818,0.590962,0.727647,"[[799, 234], [119, 255]]"
3,XGBClassifier,0.963075,0.765458,0.765458,0.593583,0.573643,0.710635,"[[855, 178], [152, 222]]"
1,KNeighborsClassifier,0.844068,0.691542,0.691542,0.684492,0.541226,0.689293,"[[717, 316], [118, 256]]"


# Apply with Scaling

In [9]:
scale_xtrain = xtrain.copy()
scale_xtest = xtest.copy()

In [10]:
sts = StandardScaler()
continuous_cols = ['tenure', 'MonthlyCharges', 'TotalCharges', 'Average']
scale_xtrain[continuous_cols] = sts.fit_transform(scale_xtrain[continuous_cols])
scale_xtest[continuous_cols] = sts.transform(scale_xtest[continuous_cols])

In [11]:
smote = SMOTE(random_state=42)
scale_xtrain_res, scale_ytrain_res = smote.fit_resample(scale_xtrain, ytrain)

In [25]:
result_list_scaled = []

for model_name, model in algorithms.items():
    model_sceled = model.fit(scale_xtrain_res, scale_ytrain_res)
    scaled_pred = model_sceled.predict(scale_xtest)

    result_list_scaled.append({
        'Model': model_name,
        'Train Score': model_sceled.score(scale_xtrain_res, scale_ytrain_res),
        'Test Score': model_sceled.score(scale_xtest, ytest),
        'Accuracy Score': accuracy_score(ytest, scaled_pred),
        'Recall Score': recall_score(ytest, scaled_pred),
        'F1 Score': f1_score(ytest, scaled_pred),
        'Roc Auc Score': roc_auc_score(ytest, scaled_pred),
        'Confussion Matrix': confusion_matrix(ytest, scaled_pred)
    })

    scaled_results = pd.DataFrame(result_list_scaled)

In [26]:
scaled_results.sort_values(by='Roc Auc Score', ascending=False)

,Model,Train Score,Test Score,Accuracy Score,Recall Score,F1 Score,Roc Auc Score,Confussion Matrix
0,LogisticRegression,0.803390,0.745558,0.745558,0.735294,0.605727,0.742284,"[[774, 259], [99, 275]]"
2,RandomForestClassifier,0.999153,0.776830,0.776830,0.657754,0.610422,0.738848,"[[847, 186], [128, 246]]"
4,GaussianNB,0.784383,0.732765,0.732765,0.748663,0.598291,0.737836,"[[751, 282], [94, 280]]"
3,XGBClassifier,0.954479,0.760483,0.760483,0.639037,0.586503,0.721745,"[[831, 202], [135, 239]]"
1,KNeighborsClassifier,0.866586,0.708600,0.708600,0.737968,0.573805,0.717968,"[[721, 312], [98, 276]]"


In [32]:
pipeline = Pipeline([
    ('smote', SMOTE(random_state=42)),
    ('model', GaussianNB())
])

gnb_param_grid = {
    'model__var_smoothing': [1e-12, 1e-10, 1e-9, 1e-8, 1e-7, 1e-6, 1e-5]
}

gnb_grid = GridSearchCV(
    estimator=pipeline, param_grid=gnb_param_grid, cv = 10, scoring='f1'
)
gnb_model = gnb_grid.fit(scale_xtrain, ytrain)

print(f"Train score: {gnb_model.score(scale_xtrain, ytrain)}")
print(f"Test score: {gnb_model.score(scale_xtest, ytest)}")

gnb_pred = gnb_model.predict(scale_xtest)

print(f"Accuracy Score: {accuracy_score(ytest, gnb_pred)}")
print(f"Recall Score: {recall_score(ytest, gnb_pred)}")
print(f"f1 Score: {f1_score(ytest, gnb_pred)}")
print(f"confussion matrix:\n{confusion_matrix(ytest, gnb_pred)}")
print(f"Best Parameter: {gnb_model.best_params_}")
print(f'Best Score: {gnb_model.best_score_}')

Train score: 0.6201209455744915
Test score: 0.5982905982905983
Accuracy Score: 0.7327647476901208
Recall Score: 0.7486631016042781
f1 Score: 0.5982905982905983
confussion matrix:
[[751 282]
 [ 94 280]]
Best Parameter: {'model__var_smoothing': 1e-12}
Best Score: 0.6172314599163327


Not good performance so we are trying logistic regession

In [34]:
logi_pipeline = Pipeline([
    ('smote', SMOTE(random_state=42)),
    ('model', LogisticRegression(max_iter=500))
])

log_param_grid = {
    'model__C': [0.01, 0.05, 0.1, 0.5, 1],
    'model__penalty': ['l1', 'l2'],
    'model__solver': ['liblinear'],
    'model__class_weight': [None, 'balanced']
}
logistic_grid = GridSearchCV(
    estimator=logi_pipeline,
    param_grid=log_param_grid,
    cv= 5, 
    scoring='f1',
    n_jobs=-1
)
logistic_model = logistic_grid.fit(scale_xtrain, ytrain)

print(f"Train Score: {logistic_model.score(scale_xtrain, ytrain)}")
print(f"Test Score: {logistic_model.score(scale_xtest, ytest)}")

logistic_pred = logistic_model.predict(scale_xtest)

print(f"confusion_matrix:\n{confusion_matrix(ytest, logistic_pred)}")
print(f"accuracy_score: {accuracy_score(ytest, logistic_pred)}")
print(f"recall_score: {recall_score(ytest, logistic_pred)}")
print(f"f1_score: {f1_score(ytest, logistic_pred)}")
print(f"Best Parameter: {logistic_model.best_params_}")
print(f'Best Score: {logistic_model.best_score_}')

Train Score: 0.6348066298342542
Test Score: 0.601081081081081
confusion_matrix:
[[760 273]
 [ 96 278]]
accuracy_score: 0.7377398720682303
recall_score: 0.7433155080213903
f1_score: 0.601081081081081
Best Parameter: {'model__C': 0.05, 'model__class_weight': None, 'model__penalty': 'l2', 'model__solver': 'liblinear'}
Best Score: 0.628414582918891


In [17]:
results = pd.DataFrame({
    'Model': ['Logistic Regression', 'KNN', 'Random Forest', 'XGBoost', 'GaussianNB', 'GaussianNB_after_tuning', 'Logistic_regression_after_tuning'],
    'Train Score': [0.8035, 0.8671, 0.9984, 0.9548, 0.7718, 0.7774, 0.8122],
    'Test Score': [0.7441, 0.7001, 0.7612, 0.7534, 0.7370, 0.6089, 0.6062],
    'Accuracy': [0.7441, 0.7001, 0.7697, 0.7534, 0.7370, 0.7370, 0.7469],
    'Recall': [0.7193, 0.7139, 0.6310, 0.6417, 0.7701, 0.7701, 0.7326],
    'F1 Score': [0.5991, 0.5586, 0.5930, 0.5804, 0.6089, 0.6089, 0.6062]
})
print("Evaluation Matrix with Scaling:")
results.sort_values(by='F1 Score', ascending=False)

Evaluation Matrix with Scaling:


,Model,Train Score,Test Score,Accuracy,Recall,F1 Score
4,GaussianNB,0.7718,0.7370,0.7370,0.7701,0.6089
5,GaussianNB_after_tuning,0.7774,0.6089,0.7370,0.7701,0.6089
6,Logistic_regression_after_tuning,0.8122,0.6062,0.7469,0.7326,0.6062
0,Logistic Regression,0.8035,0.7441,0.7441,0.7193,0.5991
2,Random Forest,0.9984,0.7612,0.7697,0.6310,0.5930
3,XGBoost,0.9548,0.7534,0.7534,0.6417,0.5804
1,KNN,0.8671,0.7001,0.7001,0.7139,0.5586
